# Notebook 02 — Limpieza, validación y preparación para SQL Server
---
* Pregunta de negocio: ¿En qué regiones, colas y horarios se presentan mayores pérdidas de servicio, y cómo se relacionan con la demanda y la disponibilidad de personal?

In [1]:
from pathlib import Path
import pandas as pd

# Apunta directamente al archivo en el directorio actual
ruta = Path("call_center_operations_wfo_dataset.csv")

df_raw = pd.read_csv(ruta)
print("Filas:", len(df_raw))
print("Columnas:", len(df_raw.columns))
df_raw.head()

Filas: 10000
Columnas: 30


,Date,Weekday,Week_Number,Month,Year,Hour,Country,Region,Site,Queue_Name,...,aHT_Seconds,handle_Time_Total,Wait_Time_Avg,shrinkage_Percent,required_FTE,agent_Count,utilization_percent,Active_Agents,Service_Level_Percent,Occupancy_Percent
0,29-09-2023,Friday,39,September,2023,3,Bulgaria,EMEA,Bulgaria Hub,INBOUND_VOICE_TIER1,...,355,171820,5.772358,28,5,6,71,4,86,89
1,27-10-2022,Thursday,43,October,2022,16,Malaysia,APAC,Kuala Lumpur,TECH_SUPPORT_EMAIL_T3,...,613,53944,78.900990,31,1,1,75,1,82,60
2,16-10-2022,Sunday,41,October,2022,15,Kenya,EMEA,Kenya Hub,EMEA_CHAT_SUPPORT_T1,...,272,17952,44.759494,18,0,0,90,0,73,71
3,23-06-2021,Wednesday,25,June,2021,12,New Zealand,APAC,New Zealand Hub,APAC_EMAIL_SUPPORT_T3,...,459,99144,6.287671,32,3,4,91,4,82,73
4,24-10-2021,Sunday,42,October,2021,18,Bangladesh,APAC,Bangladesh Hub,APAC_CORE_EMAIL_TIER2,...,867,384948,44.461538,27,13,17,71,12,77,73


In [2]:
df = df_raw.copy()
df.columns = df.columns.str.strip().str.lower()
df.insert(0, "source_row_id", range(1, len(df) + 1))

columnas_texto = ["country", "region", "site", "queue_name", "channel", "asu_tier_filter"]
for columna in columnas_texto:
    df[columna] = df[columna].str.strip()

print("Columnas después de preparar:", len(df.columns))
df.head(3)

Columnas después de preparar: 31


,source_row_id,date,weekday,week_number,month,year,hour,country,region,site,...,aht_seconds,handle_time_total,wait_time_avg,shrinkage_percent,required_fte,agent_count,utilization_percent,active_agents,service_level_percent,occupancy_percent
0,1,29-09-2023,Friday,39,September,2023,3,Bulgaria,EMEA,Bulgaria Hub,...,355,171820,5.772358,28,5,6,71,4,86,89
1,2,27-10-2022,Thursday,43,October,2022,16,Malaysia,APAC,Kuala Lumpur,...,613,53944,78.900990,31,1,1,75,1,82,60
2,3,16-10-2022,Sunday,41,October,2022,15,Kenya,EMEA,Kenya Hub,...,272,17952,44.759494,18,0,0,90,0,73,71


In [3]:
df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y", errors="coerce")

fechas_invalidas = df["date"].isna().sum()
print("Fechas inválidas:", fechas_invalidas)
print("Primera fecha:", df["date"].min())
print("Última fecha:", df["date"].max())

if fechas_invalidas > 0:
    raise ValueError("Hay fechas inválidas. Revisar el archivo antes de continuar.")

Fechas inválidas: 0
Primera fecha: 2021-01-01 00:00:00
Última fecha: 2026-12-03 00:00:00


In [4]:
df["alerta_volumen"] = df["offered_volume"] != (df["handled_volume"] + df["abandoned_volume"])
df["alerta_tiempo"] = df["handle_time_total"] != (df["handled_volume"] * df["aht_seconds"])

print("Inconsistencias de volumen:", df["alerta_volumen"].sum())
print("Inconsistencias de tiempo:", df["alerta_tiempo"].sum())

Inconsistencias de volumen: 0
Inconsistencias de tiempo: 0


In [5]:
df["alerta_personal"] = (df["agent_count"] == 0) & (df["handled_volume"] > 0)

print("Registros para revisar por personal:", df["alerta_personal"].sum())
df.loc[df["alerta_personal"], ["source_row_id", "queue_name", "handled_volume", "agent_count"]].head()

Registros para revisar por personal: 633


,source_row_id,queue_name,handled_volume,agent_count
2,3,EMEA_CHAT_SUPPORT_T1,66,0
8,9,APAC_CORE_EMAIL_T2,71,0
11,12,CUSTOMER_SUPPORT_CHAT_T1,107,0
31,32,APAC_CHAT_SUPPORT_T1,56,0
43,44,APAC_CORE_EMAIL_T2,134,0


In [6]:
df["canal_segun_cola"] = "No identificado"
df.loc[df["queue_name"].str.contains("VOICE"), "canal_segun_cola"] = "Voice"
df.loc[df["queue_name"].str.contains("CHAT"), "canal_segun_cola"] = "Chat"
df.loc[df["queue_name"].str.contains("EMAIL"), "canal_segun_cola"] = "Email"

df["validacion_canal"] = "No verificable"
df.loc[df["canal_segun_cola"] == df["channel"], "validacion_canal"] = "Coincide"
df.loc[(df["canal_segun_cola"] != "No identificado") &
       (df["canal_segun_cola"] != df["channel"]), "validacion_canal"] = "Conflicto"

print(df["validacion_canal"].value_counts())

validacion_canal
Conflicto         5980
Coincide          3034
No verificable     986
Name: count, dtype: int64


In [7]:
df["tier_segun_cola"] = "No identificado"

df.loc[df["queue_name"].str.contains("_T1") | df["queue_name"].str.contains("_TIER1"), "tier_segun_cola"] = "Tier 1"
df.loc[df["queue_name"].str.contains("_T2") | df["queue_name"].str.contains("_TIER2"), "tier_segun_cola"] = "Tier 2"
df.loc[df["queue_name"].str.contains("_T3") | df["queue_name"].str.contains("_TIER3"), "tier_segun_cola"] = "Tier 3"

df["alerta_tier"] = ((df["tier_segun_cola"] != "No identificado") &
                     (df["tier_segun_cola"] != df["asu_tier_filter"]))

print("Conflictos de Tier:", df["alerta_tier"].sum())

Conflictos de Tier: 6658


In [8]:
fecha_corte = pd.Timestamp("2026-10-29")
df["alerta_fecha_futura"] = df["date"] > fecha_corte

print("Fechas posteriores al corte:", df["alerta_fecha_futura"].sum())
print("Backlog igual a abandonos:", (df["backlog_volume"] == df["abandoned_volume"]).sum())

Fechas posteriores al corte: 30
Backlog igual a abandonos: 10000


In [9]:
df["alerta_canal"] = df["validacion_canal"] == "Conflicto"

df["requiere_revision"] = (
    df["alerta_volumen"] | df["alerta_tiempo"] |
    df["alerta_personal"] | df["alerta_canal"] |
    df["alerta_tier"] | df["alerta_fecha_futura"]
)

resumen = pd.DataFrame({
    "control": ["Volumen", "Tiempo", "Personal", "Canal", "Canal no verificable", "Tier", "Fecha futura", "Alguna alerta"],
    "registros": [df["alerta_volumen"].sum(), df["alerta_tiempo"].sum(),
                  df["alerta_personal"].sum(), df["alerta_canal"].sum(),
                  (df["validacion_canal"] == "No verificable").sum(),
                  df["alerta_tier"].sum(), df["alerta_fecha_futura"].sum(),
                  df["requiere_revision"].sum()]
})
resumen

,control,registros
0,Volumen,0
1,Tiempo,0
2,Personal,633
3,Canal,5980
4,Canal no verificable,986
5,Tier,6658
6,Fecha futura,30
7,Alguna alerta,8786


In [10]:
print("Filas originales:", len(df_raw))
print("Filas procesadas:", len(df))

for campo in ["offered_volume", "handled_volume", "abandoned_volume"]:
    total_original = df_raw[campo.replace("_volume", "_Volume")].sum()
    total_procesado = df[campo].sum()
    print(campo, ":", total_original, "→", total_procesado)
    if total_original != total_procesado:
        raise ValueError("El total de contactos cambió: " + campo)

if len(df_raw) != len(df):
    raise ValueError("El número de filas cambió durante la preparación")

Filas originales: 10000
Filas procesadas: 10000
offered_volume : 2805813 → 2805813
handled_volume : 2656676 → 2656676
abandoned_volume : 149137 → 149137


In [11]:
salida_datos = Path("../processed")
salida_reportes = Path("../reports")
if not Path("../data/raw").exists():
    salida_datos = Path("processed")
    salida_reportes = Path("reports")

salida_datos.mkdir(parents=True, exist_ok=True)
salida_reportes.mkdir(parents=True, exist_ok=True)

df.to_csv(salida_datos / "operations_clean_simple.csv", index=False, date_format="%Y-%m-%d")
resumen.to_csv(salida_reportes / "resumen_calidad_simple.csv", index=False)
print("Archivos guardados correctamente.")

Archivos guardados correctamente.
